In [2]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import requests


# PASO 1: rellenar programáticamente el selector de tipo "Registro" y un rango de fechas, ejecutar la búsqueda y capturar la URL resultante para procesarla o iterar sobre múltiples rangos de fechas de forma automática.

> El parámetro s= es un session/search ID — el frontend Vue guarda la búsqueda en el servidor y te regresa un UUID. 

> 1.1 Cada busqueda Genera un ID "SearchId" (De busqueda), en este paso tenemos que replicar la llamada API que crea ese ID. Para esto hay que identificar el flujo de request que utiliza la API (pagina >se comunica> con servidor).

    En este caso, el flujo es:
    -> Record, registra la busqueda (query parametres)
    -> Result, pagina con los registros

## Base de la funcion llamar API

In [ ]:
# session = requests.Session()
# session.get("https://marcia.impi.gob.mx/marcas/search/quick")

# # Extraes el token de las cookies que ya tienes
# xsrf_token = session.cookies.get("XSRF-TOKEN")
# print("Token:", xsrf_token)

# url = "https://marcia.impi.gob.mx/marcas/search/internal/record"

# headers = {
#     "X-XSRF-TOKEN": xsrf_token,        # ← esto es lo nuevo
#     "Content-Type": "application/json",
#     "Referer": "https://marcia.impi.gob.mx/marcas/search/quick",
# }

# payload = {
#     "_type": "Search$Structured",
#     "images": [],
#     "query": {
#         "_type": "Search$Structured",
#         "number": None,
#         "title": None,
#         "titleOption": None,
#         "name": None,
#         "status": None,
#         "appType": None,
#         "classes": None,
#         "codes": None,
#         "goodsAndServices": None,
#         "indicators": None,
#         "markType": None,
#         "wordSet": None,
#         "date": {
#             "types": ["DATE_REGISTRATION"],
#             "date": {
#                 "from": "2023-01-01",
#                 "to": "2023-01-19"
#             }
#         }
#     }
# }

# respuesta = session.post(url, json=payload, headers=headers)
# print(respuesta.status_code)
# print(respuesta.text[:500])  # solo los primeros 500 caracteres

Token: 8aba23ae-caac-4e16-a89d-e53c97266083
200
{
  "id" : "c6eb78bc-9bc4-475f-90e5-baedff97f0e1",
  "contextId" : "7494fb8d-4fea-45a9-8745-75bd9eb17bf2",
  "count" : 9263,
  "source" : "STRUCTURED",
  "query" : {
    "_type" : "Search$Structured",
    "query" : {
      "date" : {
        "types" : [ "DATE_REGISTRATION" ],
        "date" : {
          "from" : "2023-01-01",
          "to" : "2023-01-19"
        }
      }
    },
    "images" : [ ],
    "_type" : "Search$Structured"
  },
  "position" : -1
}


## Set de funciones para (llamar api en bulk)

MARCIA tiene un límite de 10,000 resultados por búsqueda. Un año completo tiene muchos más. Entonces necesitas partir el año en pedazos donde cada pedazo tenga menos de 10,000 registros, y guardar la URL de cada pedazo.

In [ ]:
# ── Sesión y autenticación ──────────────────────────────────────
session = requests.Session()
session.get("https://marcia.impi.gob.mx/marcas/search/quick")

xsrf_token = session.cookies.get("XSRF-TOKEN")
print("Token:", xsrf_token)

url = "https://marcia.impi.gob.mx/marcas/search/internal/record"

headers = {
    "X-XSRF-TOKEN": xsrf_token,
    "Content-Type": "application/json",
    "Referer": "https://marcia.impi.gob.mx/marcas/search/quick",
}

# ── Función 1: llamada a la API, funcion inicial
def llamar_api(from_date, to_date):
    payload = {
        "_type": "Search$Structured",
        "images": [],
        "query": {
            "_type": "Search$Structured",
            "number": None,
            "title": None,
            "titleOption": None,
            "name": None,
            "status": None,
            "appType": None,
            "classes": None,
            "codes": None,
            "goodsAndServices": None,
            "indicators": None,
            "markType": None,
            "wordSet": None,
            "date": {
                "types": ["DATE_REGISTRATION"],
                "date": {
                    "from": from_date.strftime("%Y-%m-%d"),
                    "to":   to_date.strftime("%Y-%m-%d")
                }
            }
        }
    }
    respuesta = session.post(url, json=payload, headers=headers)
    return respuesta.json()

#  Función 2: encontrar todos los rangos válidos de un año ─────
def encontrar_rangos(año):
    resultados = {}
    contador   = 1
    ultima_valida = None

    from_date = date(año, 1, 1)
    to_date   = date(año, 1, 1)
    fin_año   = date(año, 12, 31)

    while to_date <= fin_año:
        datos = llamar_api(from_date, to_date)
        count = datos["count"]
        print(f"  {from_date} → {to_date} | count: {count}")

        if count > 10000:
            resultados[f"llamada_{contador}"] = ultima_valida
            contador  += 1
            from_date  = to_date                  # el nuevo rango empieza aquí
            ultima_valida = None                  # limpiamos para el nuevo rango
            # no avanzamos to_date — se re-evalúa desde el mismo día

        else:
            # construir el diccionario de esta llamada válida
            search_id = datos["id"]
            ultima_valida = {
                "from":      from_date.strftime("%Y-%m-%d"),
                "to":        to_date.strftime("%Y-%m-%d"),
                "count":     count,
                "search_id": search_id,
                "url":       f"https://marcia.impi.gob.mx/marcas/search/result?s={search_id}&m=l"
            }
            to_date = to_date + timedelta(days=1)  # avanzar el cursor

    # guardar el último rango que queda al terminar el año
    if ultima_valida:
        resultados[f"llamada_{contador}"] = ultima_valida

    return resultados

# ── Ejecución ───────────────────────────────────────────────────
rangos = encontrar_rangos(2023)

print("\n── Rangos encontrados ──────────────────────────────")
for key, val in rangos.items():
    print(f"{key}: {val['from']} → {val['to']} | count: {val['count']}")
    print(f"       {val['url']}\n")

Token: e7a5fc3e-7858-4331-b65f-9b9ee9e44792
  2023-01-01 → 2023-01-01 | count: 0
  2023-01-01 → 2023-01-02 | count: 646
  2023-01-01 → 2023-01-03 | count: 1480
  2023-01-01 → 2023-01-04 | count: 2222
  2023-01-01 → 2023-01-05 | count: 3014
  2023-01-01 → 2023-01-06 | count: 3549
  2023-01-01 → 2023-01-07 | count: 3549
  2023-01-01 → 2023-01-08 | count: 3549
  2023-01-01 → 2023-01-09 | count: 4401
  2023-01-01 → 2023-01-10 | count: 4814
  2023-01-01 → 2023-01-11 | count: 5023
  2023-01-01 → 2023-01-12 | count: 5885
  2023-01-01 → 2023-01-13 | count: 6378
  2023-01-01 → 2023-01-14 | count: 6378
  2023-01-01 → 2023-01-15 | count: 6378
  2023-01-01 → 2023-01-16 | count: 7299
  2023-01-01 → 2023-01-17 | count: 8198
  2023-01-01 → 2023-01-18 | count: 8795
  2023-01-01 → 2023-01-19 | count: 9263
  2023-01-01 → 2023-01-20 | count: 10016
  2023-01-20 → 2023-01-20 | count: 753
  2023-01-20 → 2023-01-21 | count: 753
  2023-01-20 → 2023-01-22 | count: 753
  2023-01-20 → 2023-01-23 | count: 1555
  

In [ ]:
## Guardar rangos correspondientes a cada año

In [13]:
print(rangos)

{'llamada_1': {'from': '2023-01-01', 'to': '2023-01-19', 'count': 9263, 'search_id': 'ef238210-5165-497b-968e-1ab941bf7dce', 'url': 'https://marcia.impi.gob.mx/marcas/search/result?s=ef238210-5165-497b-968e-1ab941bf7dce&m=l'}, 'llamada_2': {'from': '2023-01-20', 'to': '2023-02-14', 'count': 9376, 'search_id': '8dc6156c-03c6-4369-bd04-ba78e815857e', 'url': 'https://marcia.impi.gob.mx/marcas/search/result?s=8dc6156c-03c6-4369-bd04-ba78e815857e&m=l'}, 'llamada_3': {'from': '2023-02-15', 'to': '2023-03-06', 'count': 9528, 'search_id': '4395842f-3180-4ebf-a1bf-732774c4df19', 'url': 'https://marcia.impi.gob.mx/marcas/search/result?s=4395842f-3180-4ebf-a1bf-732774c4df19&m=l'}, 'llamada_4': {'from': '2023-03-07', 'to': '2023-03-28', 'count': 9754, 'search_id': '96cc8ba4-436f-4a8e-93fb-6b4f269aec5d', 'url': 'https://marcia.impi.gob.mx/marcas/search/result?s=96cc8ba4-436f-4a8e-93fb-6b4f269aec5d&m=l'}, 'llamada_5': {'from': '2023-03-29', 'to': '2023-04-20', 'count': 9792, 'search_id': '4ddabc0b-7

In [14]:
suma = 9263 + 9376 + 9528 + 9754 + 9782 + 9938 + 9425 + 9897 + 9875 + 9450 + 9868 + 9548 + 9736 + 9909 + 8837 

print(suma)

144186


# PASO 2: Iterar lista de URLs 
